<a href="https://colab.research.google.com/github/Anuragdixit22081993/my_practice_aiml/blob/main/GenAI_Application_developer/DiffusionModels/diffusion_model_stablediffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate safetensors torch

In [ ]:
import os
from datetime import datetime

import torch
from diffusers import StableDiffusionPipeline


def create_image(
    prompt: str,
    output_folder: str = "generated_images",
    negative_prompt: str = (
        "blurry, low quality, distorted, deformed, duplicate, "
        "bad anatomy, watermark, text"
    ),
    seed: int = 42,
    inference_steps: int = 30,
    guidance_scale: float = 7.5,
) -> str:
    """
    Generate an image from a text prompt using Stable Diffusion.

    Returns:
        Path of the saved image.
    """

    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

    # Detect GPU automatically
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Using device: {device}")
    print("Loading diffusion model...")

    if device == "cuda":
        pipeline = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            use_safetensors=True,
        )

        pipeline = pipeline.to(device)

        # Reduces GPU memory consumption
        pipeline.enable_attention_slicing()

    else:
        pipeline = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
            use_safetensors=True,
        )

        pipeline = pipeline.to(device)

    # Seed makes the output reproducible
    generator = torch.Generator(device=device).manual_seed(seed)

    print(f"Generating image for prompt:\n{prompt}\n")

    result = pipeline(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        width=512,
        height=512,
    )

    image = result.images[0]

    os.makedirs(output_folder, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(
        output_folder,
        f"diffusion_image_{timestamp}.png",
    )

    image.save(output_path)

    print(f"Image generated successfully.")
    print(f"Saved at: {output_path}")

    return output_path


if __name__ == "__main__":
    user_prompt = input("Enter an image description: ")

    create_image(
        prompt=user_prompt,
        seed=42,
        inference_steps=30,
        guidance_scale=7.5,
    )